[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/personal-health-agent.github.io/blob/main/public/weeks/06-exercise.ipynb)

# Week 6 exercise: embeddings and vector search

You will turn clinical notes into embeddings with SQL, search them, change what gets embedded, and change how
the question is asked, and measure each change against notes a reader labeled as relevant. Embeddings and chat
answers come from OpenAI through the [Flock](https://duckdb.org/community_extensions/extensions/flock) DuckDB
extension; vector search uses DuckDB's [vss](https://duckdb.org/docs/current/core_extensions/vss.html) extension,
and keyword search its [fts](https://duckdb.org/docs/current/core_extensions/full_text_search.html) extension.

**How to work.** Each question has a `submit("id", """ ... """)` cell: write your SQL between the triple
quotes and run the cell. It shows the result, checks it, and saves it. Write what you found in the
`explain("id", """ ... """)` cell under it. Most cells start with a working query to edit.

**How to submit.** Submit this notebook, with its outputs: see section 6.

## 0 · Setup

1. In Colab, open the key icon (**Secrets**) in the left sidebar, add `OPENAI_API_KEY`, and turn on notebook
   access. Every section after 1 calls OpenAI; embedding all the notes costs well under a cent. The key stays
   in your Colab account and is not saved in the notebook.
2. Fill in your name and UNI below, then run the two cells. The second installs DuckDB 1.5.6 and its
   extensions and loads the notes; it takes about a minute.

In [ ]:
NAME = ""
UNI = ""
OPENAI_MODEL = "gpt-5.6-luna"                 # the chat model, for sections 5.3, 5.4, and 5.6
EMBEDDING_MODEL = "text-embedding-3-small"    # 1,536 numbers per vector

In [ ]:
# @title Setup: run this cell (you do not need to read it)
import importlib.metadata, subprocess, sys
def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None
if _version("duckdb") != "1.5.6":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.5.6"], check=True)
import duckdb
if duckdb.__version__ != "1.5.6":
    raise RuntimeError("An older DuckDB is still loaded. Use Runtime > Restart session, then run this cell again.")

"""Helpers for the Week 6 exercise notebook (week06-exercise.ipynb): embeddings and vector search.

build_notebook.py embeds this file in the notebook's setup cell, so the notebook runs on its own in
Colab. Edit this file, then rebuild the notebook. Students call only the functions listed under
"Student functions" at the end.

Embeddings and completions come from OpenAI through the Flock community extension (llm_embedding,
llm_complete, llm_filter); vector search uses DuckDB's vss extension and keyword search its fts extension.
For local testing, OPENAI_BASE_URL points Flock at a stand-in server.
"""
import datetime as _dt
import decimal as _decimal
import html as _html
import json
import os
import re

import duckdb
import pandas as pd
from IPython.display import HTML, display

# ---------- keys: Colab secrets first, then environment variables ----------
def _secret(name):
    try:
        from google.colab import userdata  # only in Colab
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name) or None

OPENAI_API_KEY = _secret("OPENAI_API_KEY")
OPENAI_MODEL = globals().get("OPENAI_MODEL") or "gpt-5.6-luna"
EMBEDDING_MODEL = globals().get("EMBEDDING_MODEL") or "text-embedding-3-small"
DIMS = int(globals().get("EMBEDDING_DIMS") or 1536)

# ---------- the database (exercise/notes.sql) ----------
DB_SQL = "-- Week 06 exercise: synthetic clinical notes, five search questions, and hand-labeled relevant notes.\n-- Every value is synthetic. Patients P001-P004 are the Week 05 patients; P005 is new.\n-- Progress notes start with the clinic's template text, which every visit repeats.\n\nCREATE TABLE patients (patient_id TEXT PRIMARY KEY, birth_year INTEGER, summary TEXT);\nINSERT INTO patients VALUES\n  ('P001', 1968, 'Type 2 diabetes, hypertension; metformin, lisinopril'),\n  ('P002', 1975, 'Type 2 diabetes; metformin'),\n  ('P003', 1990, 'Asthma; albuterol'),\n  ('P004', 1959, 'Type 2 diabetes, hypertension; insulin glargine'),\n  ('P005', 1948, 'Heart failure, COPD; furosemide, tiotropium');\n\nCREATE TABLE notes (\n  note_id    TEXT PRIMARY KEY,\n  patient_id TEXT REFERENCES patients(patient_id),\n  note_date  DATE,\n  note_type  TEXT,\n  note       TEXT\n);\n\n-- The template that begins every progress note.\nCREATE TABLE note_template AS SELECT\n  'Patient seen in clinic today for scheduled follow-up. Vital signs reviewed and recorded. Medication list reconciled with patient. Allergies reviewed and unchanged. Patient education provided and all questions answered. ' AS text;\n\nINSERT INTO notes\nSELECT note_id, patient_id, CAST(note_date AS DATE), note_type,\n       CASE WHEN note_type = 'progress note' THEN (SELECT text FROM note_template) || note ELSE note END\nFROM (VALUES\n  ('N01','P001','2025-09-01','progress note','New diagnosis of type 2 diabetes. HbA1c 8.9%. Start metformin 500 mg twice daily.'),\n  ('N02','P001','2025-09-03','nursing note','Diabetes education session completed. Reviewed glucose meter use and diet basics.'),\n  ('N03','P001','2025-09-20','patient message','I keep waking up around 3 am and cannot fall back asleep. Could that be from the new pill?'),\n  ('N04','P001','2025-09-22','progress note','Loose stools and stomach upset since starting metformin two weeks ago. Will monitor.'),\n  ('N05','P003','2025-10-02','progress note','Asthma follow-up. Uses albuterol about twice a week. Peak flow stable.'),\n  ('N06','P004','2025-10-15','phone call','Felt shaky and sweaty before lunch, meter read 58. Drank juice and felt better within 15 minutes.'),\n  ('N07','P005','2025-10-20','progress note','SOB on exertion, now after climbing one flight of stairs. Mild ankle edema. Weight up 2 kg.'),\n  ('N08','P002','2025-11-10','progress note','New diagnosis of type 2 diabetes. HbA1c 7.4%. Start metformin.'),\n  ('N09','P004','2025-11-12','progress note','Fasting glucose 182, above goal. Increase insulin glargine to 24 units at bedtime.'),\n  ('N10','P001','2025-11-18','progress note','Intermittent chest tightness when walking uphill, resolves with rest. Referred for stress test.'),\n  ('N11','P002','2025-12-01','progress note','Reports insomnia for the past month, taking one to two hours to fall asleep.'),\n  ('N12','P003','2025-12-05','progress note','Sleeps well, no daytime fatigue. Asthma well controlled.'),\n  ('N13','P001','2025-12-12','progress note','Stress test normal. No chest pain today. Heart rate regular.'),\n  ('N14','P002','2025-12-20','phone call','Nausea after the morning dose of metformin. Advised to take it with food.'),\n  ('N15','P002','2026-01-08','nursing note','Metformin refilled, 90-day supply sent to pharmacy.'),\n  ('N16','P004','2026-01-10','progress note','Diabetic foot exam: normal sensation, no ulcers.'),\n  ('N17','P004','2026-01-14','phone call','Wife reports he is up most of the night and exhausted every morning.'),\n  ('N18','P005','2026-01-20','progress note','Influenza vaccine given. COPD stable on tiotropium.'),\n  ('N19','P004','2026-02-01','progress note','Two hypoglycemic episodes this month after the insulin dose increase. Reduce glargine to 20 units.'),\n  ('N20','P003','2026-02-05','patient message','Can you refill my rescue inhaler? I am down to the last few puffs.'),\n  ('N21','P001','2026-02-10','progress note','No hypoglycemia reported. Home glucose log reviewed, readings 110 to 150.'),\n  ('N22','P003','2026-02-18','nursing note','Wheezing and winded after walking to the mailbox. Albuterol given with relief.'),\n  ('N23','P001','2026-03-01','progress note','Blood pressure 138/86 on lisinopril. Continue current dose.'),\n  ('N24','P003','2026-03-10','nursing note','States sleep is poor because of a nighttime cough; wakes several times a night.'),\n  ('N25','P001','2026-03-15','progress note','Denies shortness of breath or chest pain. HbA1c 8.2%, improving.'),\n  ('N26','P005','2026-03-20','progress note','Sleep apnea follow-up. CPAP adherence good, sleeping seven hours a night.'),\n  ('N27','P002','2026-04-02','progress note','HbA1c 7.8%. Discussed diet and walking 30 minutes a day.'),\n  ('N28','P002','2026-04-10','progress note','Stomach upset improved after switching to extended-release metformin.'),\n  ('N29','P004','2026-04-15','nursing note','Retinal photo taken for diabetic eye screening; sent to ophthalmology for review.'),\n  ('N30','P001','2026-04-20','nursing note','Continue metformin 500 mg twice daily. Next lab draw in three months.'),\n  ('N31','P005','2026-05-01','progress note','Difficulty staying asleep; averages about four hours a night.'),\n  ('N32','P003','2026-05-05','progress note','Annual physical. Feels well. Labs pending.'),\n  ('N33','P002','2026-05-12','progress note','Denies insomnia or nightmares. Mood good.'),\n  ('N34','P004','2026-05-20','progress note','LDL 148. Start atorvastatin 20 mg daily.'),\n  ('N35','P005','2026-06-02','discharge summary','Admitted for a heart failure exacerbation. Hospital course: treated with IV furosemide with good diuresis; weight down 3 kg. Dyspnea improved by day three. Medications: furosemide increased to 40 mg daily; tiotropium continued. Follow-up: heart failure clinic in one week; daily weights at home.'),\n  ('N36','P001','2026-06-10','discharge summary','Admitted for cellulitis of the left leg. Hospital course: IV cefazolin for three days, then oral cephalexin; redness and swelling resolved. Blood pressure controlled. Metformin held during the stay and restarted at discharge. Overnight on day two, glucose 62 with confusion, treated with dextrose. Follow-up: primary care in one week.'),\n  ('N37','P004','2026-06-18','phone call','Called with pressure in the chest spreading to the left arm. Advised to call 911 now.'),\n  ('N38','P003','2026-06-25','nursing note','Breathing exercises taught for anxiety before procedures.'),\n  ('N39','P005','2026-07-02','progress note','Chest X-ray clear. Lungs without crackles today.'),\n  ('N40','P002','2026-07-10','patient message','My sleep tracker says I only get about five hours. I feel wiped out all day.'),\n  ('N41','P001','2026-07-15','progress note','Weight down 3 kg since spring. Continue diet changes.'),\n  ('N42','P002','2026-07-22','patient message','Got dizzy and confused at work and my sugar was 61. I ate crackers and it passed.'),\n  ('N43','P004','2026-08-01','progress note','Blood pressure 152/94. Add amlodipine 5 mg daily.'),\n  ('N44','P001','2026-08-12','patient message','The diabetes pill leaves a metal taste in my mouth. Is that normal?'),\n  ('N45','P003','2026-08-20','progress note','Spirometry stable. Continue current inhaler plan.'),\n  ('N46','P005','2026-09-01','patient message','I get out of breath just getting dressed in the morning.')\n) AS t(note_id, patient_id, note_date, note_type, note);\n\n-- Five search questions, and the notes a reader judged relevant to each.\nCREATE TABLE search_queries (query_id TEXT PRIMARY KEY, query_text TEXT);\nINSERT INTO search_queries VALUES\n  ('Q1', 'trouble sleeping'),\n  ('Q2', 'low blood sugar episodes'),\n  ('Q3', 'side effects of metformin'),\n  ('Q4', 'shortness of breath'),\n  ('Q5', 'chest pain');\n\nCREATE TABLE relevant (query_id TEXT, note_id TEXT);\nINSERT INTO relevant VALUES\n  ('Q1','N03'), ('Q1','N11'), ('Q1','N17'), ('Q1','N24'), ('Q1','N31'), ('Q1','N40'),\n  ('Q2','N06'), ('Q2','N19'), ('Q2','N36'), ('Q2','N42'),\n  ('Q3','N04'), ('Q3','N14'), ('Q3','N28'), ('Q3','N44'),\n  ('Q4','N07'), ('Q4','N22'), ('Q4','N35'), ('Q4','N46'),\n  ('Q5','N10'), ('Q5','N37');\n"
con = duckdb.connect()
con.execute(DB_SQL)
con.execute("INSTALL vss; LOAD vss; INSTALL fts; LOAD fts;")
NOTE_TEXT = dict(con.sql("SELECT note_id, note FROM notes").fetchall())
# The note without the clinic template, for showing a note's content in a check.
NOTE_CONTENT = dict(con.sql("SELECT note_id, replace(note, (SELECT text FROM note_template), '') FROM notes").fetchall())
NOTE_PATIENT = dict(con.sql("SELECT note_id, patient_id FROM notes").fetchall())
NOTE_DATE = {k: str(v) for k, v in con.sql("SELECT note_id, note_date FROM notes").fetchall()}
QUERY_TEXT = dict(con.sql("SELECT query_id, query_text FROM search_queries").fetchall())
RELEVANT = {}
for _q, _n in con.sql("SELECT query_id, note_id FROM relevant").fetchall():
    RELEVANT.setdefault(_q, set()).add(_n)

# ---------- Flock: SQL functions that call OpenAI ----------
# The embedding model sends up to 64 texts per request. The chat model sends one row per request, so its
# call count equals the rows sent. usage_limit caps each model's tokens for the session.
BUDGET = {"total_tokens_limit": 300_000}
FLOCK_MODELS = {
    "embed": (EMBEDDING_MODEL, {"max_batch_size": 64, "usage_limit": {"total_tokens_limit": 500_000}}),
    "luna": (OPENAI_MODEL, {"max_batch_size": 1, "usage_limit": BUDGET}),
}

# Flock's Linux build looks for CA certificates at /etc/ssl/cert.pem and the Red Hat path below. Ubuntu, which
# Colab runs, keeps them at /etc/ssl/certs/ca-certificates.crt, so without these links every HTTPS call fails
# with "Empty streaming response from provider (HTTP 0)". Colab runs as root, so the links can be made.
def _link_ca_bundle():
    import platform
    bundle = "/etc/ssl/certs/ca-certificates.crt"
    if platform.system() != "Linux" or not os.path.exists(bundle):
        return
    os.environ.setdefault("SSL_CERT_FILE", bundle)
    for path in ("/etc/ssl/cert.pem", "/etc/pki/tls/certs/ca-bundle.crt"):
        if not os.path.exists(path):
            try:
                os.makedirs(os.path.dirname(path), exist_ok=True)
                os.symlink(bundle, path)
            except OSError as e:
                print(f"Could not link {path} to {bundle} ({e}); Flock's model calls may fail with HTTP 0.")

_link_ca_bundle()
con.execute("INSTALL flock FROM community; LOAD flock;")
if OPENAI_API_KEY:
    _base = os.environ.get("OPENAI_BASE_URL")
    _q = lambda v: "'" + str(v).replace("'", "''") + "'"
    con.execute(f"CREATE OR REPLACE SECRET (TYPE OPENAI, API_KEY {_q(OPENAI_API_KEY)}" + (f", BASE_URL {_q(_base)}" if _base else "") + ")")
# The models are defined even without a key, so a query fails on the missing key rather than on the model.
for _name, (_model, _args) in FLOCK_MODELS.items():
    _stmt = f"MODEL('{_name}', '{_model}', 'openai', {json.dumps(_args)})"
    try:
        con.execute("CREATE " + _stmt)
    except duckdb.Error:
        con.execute("UPDATE " + _stmt)   # the model already exists in this runtime

# Shortcuts for sections 3-5, so students write short SQL. Each wraps one Flock call.
EXPANSION_PROMPT = ("List eight different words or phrases that a patient or a clinician might write in a medical note "
                    "about this problem. Reply with the phrases only, separated by commas.")
HYPOTHETICAL_PROMPT = ("Write two sentences from a clinical note about a patient who has this problem, in the words a "
                       "clinician would use. Reply with the two sentences only.")
ANSWER_PROMPT = ("Answer the question using only the notes given. Cite the IDs of the notes you used, for example (N12). "
                 "If the notes do not answer the question, say so.")
# DuckDB evaluates a function early, while planning, when all its inputs are constants, and it may do so several
# times: SELECT embed('trouble sleeping') inside a search made three requests, and embed(expand_query(...)) eight.
# Passing each shortcut's input through at_run_time(), which DuckDB must run during the query, makes every call
# happen once per row, at run time.
NO_KEY = ("This needs an OpenAI key: add OPENAI_API_KEY in the Colab Secrets panel (the key icon), turn on notebook "
          "access, and run the setup cell again.")
con.create_function("at_run_time", lambda text: text, ["VARCHAR"], "VARCHAR", side_effects=True)
for _macro in (
    f"embed(text) AS llm_embedding({{'model_name': 'embed'}}, {{'context_columns': [{{'data': at_run_time(text)}}]}})::FLOAT[{DIMS}]",
    f"expand_query(text) AS llm_complete({{'model_name': 'luna'}}, {{'prompt': '{EXPANSION_PROMPT}', 'context_columns': [{{'data': at_run_time(text)}}]}})::VARCHAR",
    f"hypothetical_note(text) AS llm_complete({{'model_name': 'luna'}}, {{'prompt': '{HYPOTHETICAL_PROMPT}', 'context_columns': [{{'data': at_run_time(text)}}]}})::VARCHAR",
    f"answer_with(question, notes) AS llm_complete({{'model_name': 'luna'}}, {{'prompt': '{ANSWER_PROMPT}', "
    "'context_columns': [{'data': at_run_time('Question: ' || question || chr(10) || chr(10) || 'Notes:' || chr(10) || notes)}]})::VARCHAR",
):
    if OPENAI_API_KEY:
        con.execute("CREATE OR REPLACE MACRO " + _macro)
    else:   # Flock checks for its OpenAI secret when a macro is defined, so without a key each shortcut says what is missing
        con.execute(f"CREATE OR REPLACE MACRO {_macro.split(' AS ', 1)[0]} AS error('{NO_KEY}')")

def _flock_metrics():
    "Model calls, tokens, and models used per Flock function since the last reset."
    out = {}
    for key, m in json.loads(con.sql("SELECT flock_get_metrics()").fetchone()[0] or "{}").items():
        fn = re.sub(r"_\d+$", "", key)
        s = out.setdefault(fn, {"calls": 0, "input_tokens": 0, "output_tokens": 0, "models": set()})
        s["calls"] += m.get("api_calls", 0)
        s["input_tokens"] += m.get("input_tokens", 0)
        s["output_tokens"] += m.get("output_tokens", 0)
        s["models"].add(m.get("model_name"))
    return out

# ---------- questions (from build_notebook.py) and saved work ----------
QUESTIONS = {'2.1': {'label': 'One embedding', 'question': "Run the query to embed one phrase. Then change SELECT * so it returns two columns: the number of values in the vector, and the vector's length (its Euclidean norm). What do you notice about the length?"}, '2.2': {'label': 'Embed the notes', 'question': 'Embed every note into note_vectors. llm_embedding returns a list of any length; the vector functions and the index need a fixed length, so cast it to FLOAT[1536].'}, '2.3': {'label': 'Embed the questions', 'question': 'Embed the five search questions in search_queries into query_vectors (query_id, vec), the same way as the notes.'}, '2.4': {'label': 'Three measures', 'question': 'The query compares three notes pairwise by cosine similarity. Replace the two 0 placeholders with the inner product (array_inner_product) and the Euclidean distance (array_distance). How are the three numbers related, and why?'}, '3.1': {'label': 'Keyword vs vector', 'question': "Find the notes about trouble sleeping (Q1). Run the keyword search in the first cell, then the vector search in the second. The vector search first stores the question's vector in the variable q. Which relevant notes does each method miss, and why?"}, '3.2': {'label': 'Measure it in SQL', 'question': "Compute precision@5 (relevant notes in the top 5, divided by 5) and recall@10 (relevant notes in the top 10, divided by all relevant notes) for question 3.1's vector search, using the labels in relevant."}, '3.3': {'label': 'With a SQL filter', 'question': "Search only P002's notes from 2026 for Q1, and return the top 3. Add a WHERE clause to the query. Two kinds of conditions you need, shown here on other values: an equality, patient_id = 'P001', and a date comparison, note_date >= DATE '2025-12-01'. Combine conditions with AND."}, '3.4': {'label': 'Index and filter', 'question': "The first cell builds an HNSW index on note_vectors and runs question 3.3's search through it (the index serves ORDER BY a distance LIMIT k). The correct answer has 3 rows, the same as question 3.3's, but the index returns fewer. Why? Then rewrite the query in the second cell so it returns the correct 3 rows."}, '4.1': {'label': 'Remove template text', 'question': "Most progress notes begin with the same template text (see note_template). Embed the notes without it into note_vectors_clean; the check compares the five questions' scores with note_vectors. Then type your own searches into the search box below and compare the two tables' results side by side."}, '4.2': {'label': 'Add the note type', 'question': 'Embed each note with its note type in front, for example patient message: I keep waking up…, into note_vectors_typed. Then use the search box to compare note_vectors and note_vectors_typed, including a search that names a kind of note, such as “patient messages about sleep”.'}, '4.3': {'label': 'Split into sentences', 'question': 'Split each note into sentences and embed each sentence into chunk_vectors. A note then scores as its best sentence. Compare the scores, especially for Q2: N36 is a discharge summary that mentions the low glucose only in its last sentences.'}, '4.4': {'label': 'What similarity misses', 'question': 'Run the query: each row compares two short texts. Which pairs score high although they mean different things? Add a pair of your own.'}, '5.1': {'label': 'Query expansion', 'question': "expand_query(text) asks the chat model for other words and phrases for the same problem. Run the first cell to read the expansion for “trouble sleeping”, and change the text to try other questions. The second cell searches with the question stored in the variable question. Change embed(getvariable('question')) to embed(expand_query(getvariable('question'))), so it searches with the expansion instead. Did recall@10 improve over question 3.1?"}, '5.2': {'label': 'Hypothetical note', 'question': "hypothetical_note(text) asks the chat model to write a short made-up note about the problem. Searching with that note's vector instead of the question's is called hypothetical document embeddings (HyDE). Run the first cell to read a hypothetical note, and try other questions. Then change the second cell to embed(hypothetical_note(getvariable('question'))). Why can a made-up note find real notes better than the question?"}, '5.3': {'label': 'Answer with the notes (RAG)', 'question': "Retrieval-augmented generation (RAG) answers a question from retrieved text. The query finds P002's 5 notes most similar to the question. Finish it: join each note's ID and text into one string with string_agg, and pass the question and that string to answer_with(question, notes), which asks the chat model to answer using only those notes and to cite their IDs. Is the answer right, and does it cite the right notes?"}}
WORK_FILE = "week06_work.json"
try:
    with open(WORK_FILE) as _f:
        work = json.load(_f)
except (OSError, ValueError):
    work = {"student": {}, "answers": {}, "history": []}

def _save():
    with open(WORK_FILE, "w") as f:
        json.dump(work, f, default=str, indent=1)

def _record(qid, action, **data):
    t = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")
    work["history"].append({"t": t, "question": qid, "action": action, **data})
    a = work["answers"].setdefault(qid, {})
    if action == "submit":
        # The latest run replaces the saved SQL, result, and check together, so a failed or unchecked run
        # never leaves an earlier run's result or pass status attached to the new SQL.
        a["submissions"] = a.get("submissions", 0) + 1
        for k in ("sql", "result", "check", "error", "model_calls"):
            if k in data:
                a[k] = data[k]
            else:
                a.pop(k, None)
    a["updated_at"] = t
    _save()

def _need(qid):
    if qid not in QUESTIONS:
        raise ValueError(f"Unknown question {qid!r}. Use one of: {', '.join(QUESTIONS)}")

# ---------- display ----------
_COLORS = {"ok": "#e8f2ea", "bad": "#f8e9e9", "note": "#fff4e0", "info": "#e8eef5"}

def _box(kind, text):
    display(HTML(f'<div style="background:{_COLORS[kind]};border-radius:6px;padding:8px 12px;margin:6px 0;'
                 f'color:#1a1a1a;white-space:pre-wrap">{_html.escape(text)}</div>'))

def _short(v):
    "Long vectors are shown as their first values and their length."
    if isinstance(v, (list, tuple)) and len(v) > 6:
        return "[" + ", ".join(f"{x:.3f}" if isinstance(x, float) else str(x) for x in v[:4]) + f", … ({len(v)} values)]"
    return v

def _frame(cols, rows, limit=20):
    df = pd.DataFrame([[_short(v) for v in r] for r in rows[:limit]], columns=cols)
    with pd.option_context("display.max_colwidth", 120):
        display(df)
    print(f"... {len(rows)} rows in total; showing the first {limit}." if len(rows) > limit else f"{len(rows)} row(s)")

class Result:
    "One submitted statement: its output rows, the table it created, and the model calls it made."
    def __init__(self, sql, cols, rows, table, flock):
        self.sql, self.cols, self.table, self.flock = sql, list(cols or []), table, flock
        seen = {}
        for k, c in enumerate(self.cols):
            seen[c] = seen.get(c, 0) + 1
            if seen[c] > 1:
                self.cols[k] = f"{c}_{seen[c]}"   # SELECT a.note_id, b.note_id returns two columns named note_id
        # DECIMAL values (for example from ROUND on a literal) arrive as Decimal; checks compare floats.
        self.rows = [{c: float(v) if isinstance(v, _decimal.Decimal) else v for c, v in zip(self.cols, r)} for r in (rows or [])]

    def calls(self, fn):
        return self.flock.get(fn, {}).get("calls", 0)

def _summary(r):
    rows = [{k: _short(v) for k, v in x.items()} for x in r.rows[:10]]
    return {"columns": r.cols, "row_count": len(r.rows), "rows": json.loads(json.dumps(rows, default=str))}

_CREATE_AS = re.compile(r'CREATE\s+(?:OR\s+REPLACE\s+)?(?:TEMP(?:ORARY)?\s+)?TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?("[^"]+"|[\w.]+)\s+AS\b', re.I)

def _execute(sql):
    "Run a statement; if it created a table with CREATE TABLE ... AS, return that table's rows."
    if OPENAI_API_KEY:
        con.execute("SELECT flock_reset_metrics()")
    rel = con.sql(sql)
    cols, rows, table = (rel.columns, rel.fetchall(), None) if rel is not None else (None, None, None)
    flock = _flock_metrics() if OPENAI_API_KEY else {}
    if cols is None:
        made = _CREATE_AS.findall(sql)
        if made:
            table = made[-1].strip('"')
            rel = con.sql(f'SELECT * FROM {made[-1]}')
            cols, rows = rel.columns, rel.fetchall()
    return Result(sql, cols, rows, table, flock)

def _meter(r):
    return "; ".join(f"{fn}: {s['calls']} request(s), {s['input_tokens']} input and {s['output_tokens']} output tokens "
                     f"({', '.join(sorted(m for m in s['models'] if m))})" for fn, s in r.flock.items())

def _model_calls(r):
    return {fn: {"calls": s["calls"], "input_tokens": s["input_tokens"], "output_tokens": s["output_tokens"],
                 "models": sorted(m for m in s["models"] if m)} for fn, s in r.flock.items()}

# ---------- checks: shared pieces ----------
NOTE_ID = r"N\d+"
OK = lambda m, show=None: ("ok", m, show)
BAD = lambda m: ("bad", m, None)
_s = {}   # results remembered across questions, for comparisons (for example the 3.1 baseline)

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

def _near(a, b, tol=0.005):
    return _num(a) and abs(a - b) < tol

def _id_col(r, pattern=NOTE_ID):
    return next((c for c in r.cols if r.rows and all(isinstance(x[c], str) and re.fullmatch(pattern, x[c]) for x in r.rows)), None)

def _col(r, test, skip=()):
    return next((c for c in r.cols if c not in skip and any(x[c] is not None for x in r.rows)
                 and all(x[c] is None or test(x[c]) for x in r.rows)), None)

def _ranked(r):
    "The note IDs in result order, each once (several chunks of one note count once)."
    c = _id_col(r)
    out = []
    for x in r.rows if c else []:
        if x[c] not in out:
            out.append(x[c])
    return out if c else None

def _need_ids():
    return BAD("Keep the note_id column in the result, so each result can be matched to the labels.")

def metrics(ids, query_id):
    "Precision@5 and recall@10 of a ranked list of note IDs, against the hand-labeled relevant notes."
    rel = RELEVANT[query_id]
    return {"precision@5": sum(i in rel for i in ids[:5]) / 5, "recall@10": sum(i in rel for i in ids[:10]) / len(rel)}

def _fmt(m):
    return f"precision@5 {m['precision@5']:.2f}, recall@10 {m['recall@10']:.2f}"

def _labels_show(ids, query_id):
    "The top 10 next to the labels, then the relevant notes that were missed."
    rel = RELEVANT[query_id]
    rows = [{"rank": k + 1, "note_id": i, "relevant (labeled by hand)": "yes" if i in rel else "no",
             "note": NOTE_CONTENT.get(i, "")[:90]} for k, i in enumerate(ids[:10])]
    missed = [i for i in sorted(rel) if i not in ids[:10]]
    return {"rows": rows, "missed": missed, "query_id": query_id}

def _show_labels(show):
    with pd.option_context("display.max_colwidth", 100):
        display(pd.DataFrame(show["rows"]))
    if show["missed"]:
        _box("note", f"Relevant notes for {show['query_id']} that are not in your top 10: " +
             "; ".join(f"{i} (“{NOTE_CONTENT[i][:90].rstrip('.')}”)" for i in show["missed"]) +
             ". The labels are a reader's judgment of each note; say in your explain cell whether you agree.")

def _search_check(r, query_id, label, baseline_key=None, remember=None):
    "Score a ranked result against the labels, and compare with an earlier result if there is one."
    ids = _ranked(r)
    if ids is None:
        return _need_ids()
    if not ids:
        return BAD("The search returned no notes.")
    m = metrics(ids, query_id)
    if remember:
        _s[remember] = m
    msg = f"{label}: {_fmt(m)} for {query_id} (“{QUERY_TEXT[query_id]}”)."
    base = _s.get(baseline_key) if baseline_key else None
    if base:
        msg += f" Your vector search in question 3.1 had {_fmt(base)}."
    return OK(msg, _labels_show(ids, query_id))

def _table_info(name):
    "Rows, the vector column, and its dimension for a table, or None if it does not exist."
    try:
        cols = con.sql(f"DESCRIBE {name}").fetchall()
    except duckdb.Error:
        return None
    vec = next(((c, t) for c, t, *_ in cols if re.fullmatch(r"FLOAT\[\d+\]", t)), None)
    n = con.sql(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    return {"rows": n, "cols": [c for c, *_ in cols], "vec": vec[0] if vec else None,
            "dims": int(re.search(r"\d+", vec[1]).group()) if vec else None,
            "types": {c: t for c, t, *_ in cols}}

def evaluate(note_table="note_vectors", query_table="query_vectors", show=True):
    """Search every question in query_table against note_table (cosine similarity, exact) and score the top 10.

    A note with several rows (chunks) gets its best chunk's score. Returns one row per question."""
    nt, qt = _table_info(note_table), _table_info(query_table)
    for name, t in ((note_table, nt), (query_table, qt)):
        if t is None:
            raise ValueError(f"There is no table {name}.")
        if not t["vec"]:
            raise ValueError(f"{name} has no fixed-length vector column (FLOAT[n]).")
    if nt["dims"] != qt["dims"]:
        raise ValueError(f"{note_table} has {nt['dims']}-dimensional vectors but {query_table} has {qt['dims']}: "
                         "notes and questions must be embedded the same way.")
    scored = con.sql(f"""SELECT q.query_id, n.note_id, max(array_cosine_similarity(n.{nt['vec']}, q.{qt['vec']})) AS s
                         FROM {note_table} AS n, {query_table} AS q GROUP BY ALL ORDER BY q.query_id, s DESC""").fetchall()
    ranked = {}
    for q, n, _ in scored:
        ranked.setdefault(q, []).append(n)
    out = pd.DataFrame([{"query": q, "text": QUERY_TEXT.get(q, ""), **metrics(ids, q)} for q, ids in sorted(ranked.items()) if q in RELEVANT])
    if show:
        display(out)
    return out

def _compare_tables(new_table, new_query_table="query_vectors", label=""):
    "evaluate() a new table against note_vectors; a message and the side-by-side scores."
    base = evaluate("note_vectors", "query_vectors", show=False)
    new = evaluate(new_table, new_query_table, show=False)
    both = base.merge(new, on=["query", "text"], suffixes=(" before", " after"))
    display(both)
    b, a = base["recall@10"].mean(), new["recall@10"].mean()
    return f"{label} Mean recall@10 over the five questions: {b:.2f} with note_vectors, {a:.2f} with {new_table}."

def _needs_table(name, rows=None, dims=None):
    t = _table_info(name)
    if t is None:
        return None, BAD(f"Create the table {name}, as the question asks.")
    if not t["vec"]:
        kind = t["types"].get("vec", "a list")
        return None, BAD(f"{name}.vec is {kind}. Cast it to a fixed-length vector, for example ::FLOAT[{dims or DIMS}]: "
                         "the vector functions and the index need a fixed length.")
    if rows is not None and t["rows"] != rows:
        return None, BAD(f"{name} has {t['rows']} rows; expected {rows}.")
    if dims is not None and t["dims"] != dims:
        return None, BAD(f"{name} has {t['dims']}-dimensional vectors; expected {dims}.")
    if "note_id" not in t["cols"] and "query_id" not in t["cols"]:
        return None, BAD(f"Keep the note_id (or query_id) column in {name}.")
    return t, None

# ---------- section 2: creating embeddings ----------
def _check_2_1(r):
    if not r.calls("llm_embedding"):
        return BAD("This question needs llm_embedding.")
    nums = [v for x in r.rows for v in x.values() if _num(v)]
    has_len, has_norm = DIMS in nums, any(_near(v, 1.0, 0.01) for v in nums)
    if has_len and has_norm:
        return OK(f"The vector has {DIMS} numbers and length 1.0: the model returns normalized vectors.")
    if has_len:
        return BAD(f"The vector has {DIMS} numbers. Now add its length: sqrt(list_sum(list_transform(vec, x -> x * x))).")
    return BAD("Add the number of values (len(vec)) and the vector's length (its Euclidean norm) as columns.")

def _check_2_2(r):
    t, err = _needs_table("note_vectors", rows=len(NOTE_TEXT), dims=DIMS)
    if err:
        return err
    if not {"patient_id", "note_date"} <= set(t["cols"]):
        return BAD("Keep patient_id and note_date in note_vectors; questions 3.3 and 3.4 filter on them.")
    if con.sql("SELECT COUNT(*) FROM note_vectors WHERE vec IS NULL").fetchone()[0]:
        return BAD("Some rows have no vector. Check that every note was embedded.")
    calls = r.calls("llm_embedding")
    return OK(f"note_vectors holds {t['rows']} notes as {DIMS}-dimensional vectors" +
              (f", embedded in {calls} request(s) of up to 64 notes." if calls else "."))

def _check_2_3(r):
    t, err = _needs_table("query_vectors", rows=len(QUERY_TEXT), dims=DIMS)
    if err:
        return err
    return OK(f"query_vectors holds the {t['rows']} search questions, embedded the same way as the notes.")

def _check_2_4(r):
    ids = [c for c in r.cols if r.rows and all(isinstance(x[c], str) and re.fullmatch(NOTE_ID, x[c]) for x in r.rows)]
    if len(ids) < 2 or len(r.rows) < 3:
        return BAD("Keep both note IDs (note_a, note_b) and the three pairs of N03, N04, and N11.")
    true = {(a, b): v for a, b, *v in con.sql("""SELECT a.note_id, b.note_id, array_cosine_similarity(a.vec, b.vec),
        array_inner_product(a.vec, b.vec), array_distance(a.vec, b.vec) FROM note_vectors a, note_vectors b""").fetchall()}
    def matches(k):   # k = 0 cosine, 1 inner product, 2 Euclidean distance
        return [c for c in r.cols if all(_num(x[c]) and abs(x[c] - true[(x[ids[0]], x[ids[1]])][k]) < 0.001 for x in r.rows)]
    dot = [c for c in matches(1) if c != "cosine"]
    dist = matches(2)
    if not dot:
        return BAD("Replace the first 0 with the inner product: array_inner_product(a.vec, b.vec).")
    if not dist:
        return BAD("Replace the second 0 with the Euclidean distance: array_distance(a.vec, b.vec).")
    return OK(f"For every pair, the inner product ({dot[0]}) equals the cosine similarity, and the squared Euclidean distance "
              f"({dist[0]}) equals 2 − 2 × cosine. All three give the same ranking. Why?")

# ---------- section 3: searching ----------
def _q_vector():
    try:
        return con.sql("SELECT getvariable('q')").fetchone()[0]
    except duckdb.Error:
        return None

def _check_3_1(r):
    if not r.calls("llm_embedding"):
        ids = _ranked(r)
        if ids is None:
            return _need_ids()
        rel = RELEVANT["Q1"]
        _s["keyword"] = ids
        found = [i for i in ids if i in rel]
        return OK(f"Keyword search: {len(ids)} notes, {len(found)} of the {len(rel)} relevant ones. "
                  "Now run the vector search in the next cell.", _labels_show(ids, "Q1"))
    return _search_check(r, "Q1", "Vector search", remember="Q1")

def _check_3_2(r):
    q = _q_vector()
    if q is None:
        return BAD("Run question 3.1's vector search first, so the variable q holds the question's vector.")
    ids = [n for n, in con.sql("SELECT note_id FROM note_vectors ORDER BY array_cosine_similarity(vec, getvariable('q')) DESC").fetchall()]
    want = metrics(ids, "Q1")
    nums = [v for x in r.rows for v in x.values() if _num(v)]
    p = any(_near(v, want["precision@5"], 0.001) for v in nums)
    rc = any(_near(v, want["recall@10"], 0.001) for v in nums)
    if p and rc:
        return OK(f"Correct: {_fmt(want)}, the same numbers the checker computes.")
    if p:
        return BAD(f"Precision@5 is right ({want['precision@5']:.2f}). Recall@10 divides the relevant notes in the top 10 "
                   "by all relevant notes for Q1.")
    if rc:
        return BAD(f"Recall@10 is right ({want['recall@10']:.2f}). Precision@5 counts the relevant notes in the top 5, "
                   "divided by 5.")
    return BAD(f"Expected precision@5 {want['precision@5']:.2f} and recall@10 {want['recall@10']:.2f}. Count relevant notes "
               "with rank <= 5 (divided by 5) and with rank <= 10 (divided by the number of relevant notes).")

def _exact_filtered(k=3):
    return [n for n, in con.sql(f"""SELECT note_id FROM note_vectors WHERE patient_id = 'P002' AND note_date >= DATE '2026-01-01'
                                    ORDER BY array_cosine_similarity(vec, getvariable('q')) DESC LIMIT {k}""").fetchall()]

def _check_3_3(r):
    if _q_vector() is None:
        return BAD("Run question 3.1's vector search first, so the variable q holds the question's vector.")
    ids = _ranked(r) if r.rows else []
    if r.rows and ids is None:
        return _need_ids()
    wrong = [i for i in ids if NOTE_PATIENT.get(i) != "P002" or NOTE_DATE.get(i, "") < "2026-01-01"]
    if wrong:
        return BAD(f"{', '.join(wrong)} {'is not a 2026 note' if len(wrong) == 1 else 'are not 2026 notes'} of P002. "
                   "Filter on patient_id and note_date in WHERE.")
    want = _exact_filtered()
    if ids != want:
        return BAD(f"Expected the 3 most similar of P002's 2026 notes, most similar first; your result has {len(ids)}.")
    return OK(f"Correct: {', '.join(ids)}. The filter ran before the ranking, so these are the true top 3 of P002's 2026 notes.")

def _check_3_4(r):
    has_index = bool(con.sql("SELECT COUNT(*) FROM duckdb_indexes() WHERE table_name = 'note_vectors'").fetchone()[0])
    if not has_index:
        return BAD("Create the HNSW index on note_vectors first (the first cell).")
    ids = _ranked(r) if r.rows else []
    if r.rows and ids is None:
        return _need_ids()
    want = _exact_filtered()
    if ids == want:
        return OK(f"Correct: {', '.join(ids)}, the same 3 notes as question 3.3, with the index in place.")
    if len(ids) < 3:
        return BAD(f"This query returned {len(ids)} row(s), but the correct answer has 3: the 3 most similar of P002's 2026 "
                   "notes, as in question 3.3. The index found the nearest notes of all patients before WHERE removed the "
                   "others. Explain why, then rewrite the query in the second cell.")
    return BAD("Expected the same 3 notes as question 3.3, most similar first.")

# ---------- section 4: controlling the embeddings ----------
def _check_4_1(r):
    t, err = _needs_table("note_vectors_clean", rows=len(NOTE_TEXT), dims=DIMS)
    if err:
        return err
    if not r.calls("llm_embedding"):
        return BAD("Embed the notes again, with the template text removed, using llm_embedding.")
    same = con.sql("SELECT COUNT(*) FROM note_vectors a JOIN note_vectors_clean b USING (note_id) "
                   "WHERE array_cosine_similarity(a.vec, b.vec) > 0.9999").fetchone()[0]
    if same == len(NOTE_TEXT):
        return BAD("Every vector is the same as in note_vectors: the embedded text did not change. Remove the template text "
                   "with replace(note, (SELECT text FROM note_template), '').")
    return OK(_compare_tables("note_vectors_clean", label="Template text removed."))

def _check_4_2(r):
    t, err = _needs_table("note_vectors_typed", rows=len(NOTE_TEXT), dims=DIMS)
    if err:
        return err
    if not r.calls("llm_embedding"):
        return BAD("Embed the notes again, with the note type added to the text, using llm_embedding.")
    return OK(_compare_tables("note_vectors_typed", label="Note type added to the embedded text."))

def _check_4_3(r):
    t, err = _needs_table("chunk_vectors", dims=DIMS)
    if err:
        return err
    if t["rows"] <= len(NOTE_TEXT):
        return BAD(f"chunk_vectors has {t['rows']} rows, no more than the {len(NOTE_TEXT)} notes: split each note into "
                   "sentences before embedding.")
    if "note_id" not in t["cols"]:
        return BAD("Keep note_id in chunk_vectors, so each chunk points back to its note.")
    msg = _compare_tables("chunk_vectors", label=f"{t['rows']} chunks; each note scored by its best chunk.")
    rank = lambda table: next((k + 1 for k, (n,) in enumerate(con.sql(
        f"""SELECT n.note_id FROM {table} AS n, query_vectors AS q WHERE q.query_id = 'Q2'
            GROUP BY n.note_id ORDER BY max(array_cosine_similarity(n.vec, q.vec)) DESC""").fetchall()) if n == "N36"), None)
    return OK(msg + f" N36, a discharge summary whose low glucose is in its last sentences, ranks {rank('note_vectors')} "
              f"for Q2 as one vector and {rank('chunk_vectors')} as chunks.")

def _check_4_4(r):
    if not r.calls("llm_embedding"):
        return BAD("This question needs llm_embedding on both texts of each pair.")
    c = _col(r, lambda v: _num(v) and -1.0001 <= v <= 1.0001)
    if not c:
        return BAD("Return the cosine similarity of each pair.")
    return OK("Similarities computed. Which pairs score high although they mean different things? Answer in the explain cell.")

# ---------- section 5: changing the query, and RAG ----------
def _variable_text(name):
    try:
        return con.sql(f"SELECT getvariable('{name}')").fetchone()[0]
    except duckdb.Error:
        return None

def _uses(sql, name):
    return re.search(r"getvariable\(\s*'" + name + r"'\s*\)", sql, re.I) is not None

def _llm_search_check(r, shortcut, what, label):
    "First cell: read the model's text. Second cell: search with the vector of the model's text for the question."
    if not r.calls("llm_embedding"):
        if not r.calls("llm_complete"):
            return BAD(f"This cell needs {shortcut}(...).")
        return OK(f"Read the {what}. Change the text in this cell to try other questions; then run the search in the "
                  "next cell.")
    if not r.calls("llm_complete") or shortcut not in r.sql:
        return BAD(f"The search still embeds the question itself. Wrap it in {shortcut}: "
                   f"embed({shortcut}(getvariable('question'))).")
    question = _variable_text("question")
    if (question or "").strip().lower() != QUERY_TEXT["Q1"]:
        return OK(f"Searched with the {what} for “{question}”. Only “{QUERY_TEXT['Q1']}” has labeled notes, so set the "
                  "question back to it to see the scores.")
    return _search_check(r, "Q1", label, baseline_key="Q1")

def _check_5_1(r):
    return _llm_search_check(r, "expand_query", "expansion", "Expanded question")

def _check_5_2(r):
    return _llm_search_check(r, "hypothetical_note", "hypothetical note", "Hypothetical-note search (HyDE)")

# P002's notes that say the patient has trouble sleeping (N33 says the patient denies insomnia).
RAG_RELEVANT = {"N11", "N40"}

def _check_5_3(r):
    if not r.calls("llm_complete"):
        return BAD("Finish the query with answer_with(question, string_agg(...)) so the chat model answers from the notes.")
    if r.calls("llm_complete") > 1:
        return BAD(f"answer_with was called {r.calls('llm_complete')} times. Join the notes into one string with string_agg, "
                   "so the model gets all of them in one call.")
    texts = [v for x in r.rows for v in x.values() if isinstance(v, str)]
    if not texts:
        return BAD("Return the model's answer as a column.")
    answer = max(texts, key=len)
    cited = sorted(set(re.findall(r"\bN\d{2}\b", answer)))
    other = [i for i in cited if NOTE_PATIENT.get(i) != "P002"]
    if other:
        return BAD(f"The answer cites {', '.join(other)}, which belong to other patients. Keep WHERE v.patient_id = 'P002' "
                   "in the retrieval.")
    msg = (f"The answer cites {', '.join(cited)}." if cited else "The answer cites no note IDs.") + \
          f" A reader would cite {', '.join(sorted(RAG_RELEVANT))} (N33 says the patient denies insomnia)."
    missing = sorted(RAG_RELEVANT - set(cited))
    if missing:
        msg += f" Not cited: {', '.join(missing)}. Were they among the 5 retrieved notes?"
    return OK(msg)

CHECKS = {
    "2.1": _check_2_1, "2.2": _check_2_2, "2.3": _check_2_3, "2.4": _check_2_4,
    "3.1": _check_3_1, "3.2": _check_3_2, "3.3": _check_3_3, "3.4": _check_3_4,
    "4.1": _check_4_1, "4.2": _check_4_2, "4.3": _check_4_3, "4.4": _check_4_4,
    "5.1": _check_5_1, "5.2": _check_5_2, "5.3": _check_5_3,
}

# ---------- Student functions ----------
def show_table(name, n=5):
    "Show a table's row count and its first n rows."
    total = con.sql(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    rel = con.sql(f"SELECT * FROM {name} LIMIT {int(n)}")
    print(f"{name}: {total} rows in total" + (f"; showing the first {n}" if total > n else ""))
    _frame(rel.columns, rel.fetchall(), n)

def submit(qid, sql):
    "Run your SQL for a question, show the result, check it, and save it."
    _need(qid)
    try:
        r = _execute(sql)
    except Exception as e:   # DuckDB errors, and errors Flock raises from its model calls
        msg = f"{type(e).__name__}: {e}"
        if not OPENAI_API_KEY and re.search(r"llm_|Secret not found", str(e)):
            msg = "Flock needs an OpenAI key: add OPENAI_API_KEY in Colab's Secrets panel, run the setup cell again, then this cell.\n" + msg
        elif "Empty streaming response" in str(e) or "HTTP 0" in str(e):
            msg = ("The request never reached OpenAI (no HTTP response), so this is a connection problem, not your SQL. "
                   "Run the setup cell again, then this cell; if it persists, tell the course staff.\n" + msg)
        elif "Output extraction error" in str(e) or "Failed to parse" in str(e):
            msg = ("OpenAI sent back an error instead of an answer. Check that OPENAI_API_KEY is valid and that your key "
                   "can use the model.\n" + msg)
        elif re.search(r"Cannot cast list with length|Array size", str(e)):
            msg = "Two vectors of different lengths were compared, or a list was cast to the wrong FLOAT[n]. Check the dimensions.\n" + msg
        _box("bad", msg)
        _record(qid, "submit", sql=sql, error=str(e))
        return
    if r.table:
        print(f"Created table {r.table}:")
    if r.cols:
        _frame(r.cols, [tuple(x.values()) for x in r.rows])
    else:
        print("Done.")
    if _meter(r):
        print(_meter(r))
    verdict = None
    if qid in CHECKS:
        try:
            status, message, show = CHECKS[qid](r)
        except Exception as e:
            status, message, show = "bad", f"Could not check this result: {e}", None
        _box(status, f"Question {qid}: {message}")
        if show:
            _show_labels(show)
        verdict = {"status": status, "message": message}
    _record(qid, "submit", sql=sql, result=_summary(r), model_calls=_model_calls(r), **({"check": verdict} if verdict else {}))

def run(sql):
    "Run any query and show its result, without checking it. The query is saved in your history."
    try:
        r = _execute(sql)
    except Exception as e:
        _box("bad", f"{type(e).__name__}: {e}")
        work["history"].append({"t": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"),
                                "question": None, "action": "run", "sql": sql, "error": str(e)})
        _save()
        return
    if r.table:
        print(f"Created table {r.table}:")
    if r.cols:
        _frame(r.cols, [tuple(x.values()) for x in r.rows])
    else:
        print("Done.")
    if _meter(r):
        print(_meter(r))
    work["history"].append({"t": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"), "question": None,
                            "action": "run", "sql": sql, "result": _summary(r), "model_calls": _model_calls(r)})
    _save()

def compare_search(text, table_a="note_vectors", table_b="note_vectors_clean", k=5, qid=None):
    "Embed one search and show the top k notes from two vector tables side by side."
    for t in (table_a, table_b):
        if _table_info(t) is None:
            _box("bad", f"There is no table {t} yet. Run the question's submit cell first.")
            return
    lit = "'" + str(text).replace("'", "''") + "'"
    con.execute(f"SET VARIABLE search_box = (SELECT embed({lit}))")
    cols = []
    for t in (table_a, table_b):
        rows = con.sql(f"""SELECT note_id, round(max(array_cosine_similarity(vec, getvariable('search_box'))), 3) AS similarity
                           FROM {t} GROUP BY note_id ORDER BY similarity DESC LIMIT {int(k)}""").fetchall()
        cols.append(pd.DataFrame([{"note_id": n, "similarity": s, "note": NOTE_CONTENT[n][:70]} for n, s in rows]))
    parts = "".join(f'<div style="flex:1;min-width:0"><b>{_html.escape(t)}</b>{df.to_html(index=False)}</div>'
                    for t, df in zip((table_a, table_b), cols))
    display(HTML(f'<div style="display:flex;gap:16px">{parts}</div>'))
    if qid:
        work["history"].append({"t": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"), "question": qid,
                                "action": "search", "text": text, "tables": [table_a, table_b],
                                "results": {t: list(df["note_id"]) for t, df in zip((table_a, table_b), cols)}})
        _save()

def search_box(qid, table_a, table_b):
    "A text box and a Search button: each search shows the top 5 notes from both tables side by side."
    try:
        import ipywidgets as w
    except ImportError:
        print(f"No search box here: call compare_search('your search', '{table_a}', '{table_b}') instead.")
        return
    box = w.Text(placeholder="Type a search, for example: nausea", layout=w.Layout(width="60%"))
    button, out = w.Button(description="Search"), w.Output()
    def go(_):
        out.clear_output()
        with out:
            if box.value.strip():
                compare_search(box.value.strip(), table_a, table_b, qid=qid)
    button.on_click(go)
    display(w.HBox([box, button]), out)

def explain(qid, text):
    "Save your written answer for a question."
    _need(qid)
    work["answers"].setdefault(qid, {})["answer"] = text.strip()
    _record(qid, "answer", answer=text.strip())
    print(f"Saved your answer for question {qid}.")

def show_schema():
    "Show each table's columns and types."
    display(pd.DataFrame(con.sql("""SELECT table_name, string_agg(column_name || ' ' || data_type, ', ' ORDER BY column_index)
        FROM duckdb_columns() WHERE NOT internal AND schema_name = 'main' GROUP BY table_name
        ORDER BY array_position(['patients', 'notes', 'note_template', 'search_queries', 'relevant'], table_name)""").fetchall(),
        columns=["table", "columns"]))

def progress():
    "Show what you have submitted so far."
    rows = []
    for qid, q in QUESTIONS.items():
        a = work["answers"].get(qid, {})
        chk = a.get("check")
        rows.append({"question": qid, "": q["label"], "submissions": a.get("submissions", 0),
                     "last check": {"ok": "pass", "bad": "fail"}.get(chk["status"], "") if chk else "",
                     "written answer": "yes" if a.get("answer") else ""})
    display(pd.DataFrame(rows))
    print(f"{len(work['history'])} entries in the history.")

print(f"DuckDB {duckdb.__version__} is ready with tables: patients, notes, note_template, search_queries, relevant.")
print("OpenAI:", f"key found; Flock models embed ({EMBEDDING_MODEL}, {DIMS} dimensions) and luna ({OPENAI_MODEL})"
      if OPENAI_API_KEY else "no key. Add OPENAI_API_KEY in Colab's Secrets panel: every section after 1 needs it.")
if work["history"]:
    print(f"Restored your saved work: {len(work['history'])} history entries. Tables are not restored: rerun the cells that create them.")

## 1 · The notes and the questions

`notes` holds 46 synthetic notes for five patients: progress notes, nursing notes, phone calls, patient
messages, and discharge summaries. Most progress notes begin with the same clinic template (`note_template`).
`search_queries` holds five search questions, and `relevant` lists the notes a reader judged relevant to each.
Every search below is scored against those labels with **precision@5** (how many of the top 5 are relevant)
and **recall@10** (how many of the relevant notes appear in the top 10). The labels are one reader's judgment;
disagree in your explain cells when you have a reason.

In [ ]:
show_schema()

In [ ]:
show_table("notes", 8)

In [ ]:
show_table("search_queries")

**Try any query.** `run(sql)` runs a query and shows its result without checking it, for example to look at a
table or test part of an answer. It is saved in your history. Add a cell with it anywhere in the notebook.

In [ ]:
run("""
SELECT note_id, note_type, note
FROM notes
WHERE patient_id = 'P002';
""")

## 2 · Creating embeddings

`llm_embedding` sends text to the embedding model and returns its vector:

```sql
llm_embedding({'model_name': 'embed'}, {'context_columns': [{'data': <text>}]})   -- a list of 1,536 numbers
```

Flock joins the values in `context_columns` with spaces, so whatever text you build in SQL is what gets
embedded. Under each result, `submit` prints the requests and tokens the statement used. Flock does not
cache: running a statement again embeds the text again.

**Question 2.1 · One embedding.** Run the query to embed one phrase. Then change `SELECT *` so it returns two columns: the number of values in the vector, and the vector's length (its Euclidean norm). What do you notice about the length?

<details><summary>Hint</summary>

Replace `SELECT *` with `SELECT len(vec) AS n_values, sqrt(list_sum(list_transform(vec, x -> x * x))) AS length`. `list_transform` squares each value, `list_sum` adds them up, and `sqrt` takes the square root.
</details>

In [ ]:
submit("2.1", """
WITH embed AS (
  SELECT llm_embedding({'model_name': 'embed'},
                       {'context_columns': [{'data': 'trouble sleeping'}]}) AS vec
)
SELECT * FROM embed;
""")

In [ ]:
explain("2.1", """

""")   # What is the vector's length, and why might a model return vectors of that length?

**Question 2.2 · Embed the notes.** Embed every note into `note_vectors`. `llm_embedding` returns a list of any length; the vector functions and the index need a fixed length, so cast it to `FLOAT[1536]`.

<details><summary>Hint</summary>

Add `::FLOAT[1536]` right after the `llm_embedding(...)` call. Keep `patient_id` and `note_date`: questions 3.3 and 3.4 filter on them.
</details>

In [ ]:
submit("2.2", """
CREATE OR REPLACE TABLE note_vectors AS
SELECT note_id, patient_id, note_date,
       llm_embedding({'model_name': 'embed'}, {'context_columns': [{'data': note}]}) AS vec
FROM notes;
""")

In [ ]:
explain("2.2", """

""")   # How many requests did embedding all 46 notes take, and why not 46?

**Question 2.3 · Embed the questions.** Embed the five search questions in `search_queries` into `query_vectors` (`query_id`, `vec`), the same way as the notes.

<details><summary>Hint</summary>

Copy question 2.2's query. The text column here is `query_text`.
</details>

In [ ]:
submit("2.3", """
CREATE OR REPLACE TABLE query_vectors AS
SELECT query_id,
       ... AS vec
FROM search_queries;
""")

In [ ]:
explain("2.3", """

""")   # Why must the questions be embedded with the same model as the notes?

**Question 2.4 · Three measures.** The query compares three notes pairwise by cosine similarity. Replace the two `0` placeholders with the inner product (`array_inner_product`) and the Euclidean distance (`array_distance`). How are the three numbers related, and why?

<details><summary>Hint</summary>

`array_inner_product(a.vec, b.vec)` and `array_distance(a.vec, b.vec)`. Question 2.1 showed each vector's length: for vectors of length 1, what does the cosine formula reduce to?
</details>

In [ ]:
submit("2.4", """
WITH v AS (
  SELECT * FROM note_vectors WHERE note_id IN ('N03', 'N04', 'N11')
)
SELECT a.note_id AS note_a, b.note_id AS note_b,
       array_cosine_similarity(a.vec, b.vec) AS cosine,
       0 AS inner_product,
       0 AS euclidean
FROM v AS a JOIN v AS b ON a.note_id < b.note_id;
""")

In [ ]:
explain("2.4", """

""")   # How are cosine similarity, inner product, and Euclidean distance related here, and why?

## 3 · Searching

A vector search embeds the question, compares its vector with every note's vector, and returns the most
similar notes:

```sql
array_cosine_similarity(a, b)   -- 1 for the same direction; higher is more similar
array_cosine_distance(a, b)     -- 1 − similarity; lower is more similar (what the index uses)
```

From here on, `embed(text)` is a shorthand that the setup cell defined for the call you wrote in section 2:
`llm_embedding({'model_name': 'embed'}, {'context_columns': [{'data': text}]})::FLOAT[1536]`.
`SET VARIABLE q = (SELECT embed('...'))` embeds the question once and stores its vector, and `getvariable('q')`
uses it in the query.

**Question 3.1 · Keyword vs vector.** Find the notes about trouble sleeping (Q1). Run the keyword search in the first cell, then the vector search in the second. The vector search first stores the question's vector in the variable `q`. Which relevant notes does each method miss, and why?

<details><summary>Hint</summary>

Compare the wording of the missed notes with the search words. Look also at notes the keyword search returns that are not about trouble sleeping.
</details>

In [ ]:
submit("3.1", """
SELECT note_id, note
FROM notes
WHERE note ILIKE '%sleep%';
""")

In [ ]:
submit("3.1", """
SET VARIABLE q = (SELECT embed('trouble sleeping'));

SELECT n.note_id, n.note,
       array_cosine_similarity(v.vec, getvariable('q')) AS similarity
FROM note_vectors AS v JOIN notes AS n USING (note_id)
ORDER BY similarity DESC
LIMIT 10;
""")

In [ ]:
explain("3.1", """

""")   # Explain the difference between the two searches: which notes does each find or miss, and why?

**Question 3.2 · Measure it in SQL.** Compute precision@5 (relevant notes in the top 5, divided by 5) and recall@10 (relevant notes in the top 10, divided by all relevant notes) for question 3.1's vector search, using the labels in `relevant`.

<details><summary>Hint</summary>

`count(relevant.note_id) FILTER (WHERE rank <= 5)` counts the relevant notes in the top 5. The number of relevant notes for Q1 is `(SELECT count(*) FROM relevant WHERE query_id = 'Q1')`.
</details>

In [ ]:
submit("3.2", """
WITH ranked AS (
  SELECT note_id,
         row_number() OVER (ORDER BY array_cosine_similarity(vec, getvariable('q')) DESC) AS rank
  FROM note_vectors
)
SELECT ... AS precision_at_5,
       ... AS recall_at_10
FROM ranked
LEFT JOIN relevant ON relevant.query_id = 'Q1' AND relevant.note_id = ranked.note_id;
""")

In [ ]:
explain("3.2", """

""")   # What do precision@5 and recall@10 each tell you that the other does not?

**Question 3.3 · With a SQL filter.** Search only **P002's notes from 2026** for Q1, and return the top 3. Add a `WHERE` clause to the query. Two kinds of conditions you need, shown here on other values: an equality, `patient_id = 'P001'`, and a date comparison, `note_date >= DATE '2025-12-01'`. Combine conditions with `AND`.

<details><summary>Hint</summary>

`WHERE` goes after `FROM` and before `ORDER BY`. The filter runs first; the ranking then orders only the rows that pass it.
</details>

In [ ]:
submit("3.3", """
SELECT note_id, patient_id, note_date,
       array_cosine_similarity(vec, getvariable('q')) AS similarity
FROM note_vectors
ORDER BY similarity DESC
LIMIT 3;
""")

In [ ]:
explain("3.3", """

""")   # Why must a filter such as the patient run before the ranking, not after?

**Question 3.4 · Index and filter.** The first cell builds an HNSW index on `note_vectors` and runs question 3.3's search through it (the index serves `ORDER BY` a distance `LIMIT k`). The correct answer has 3 rows, the same as question 3.3's, but the index returns fewer. Why? Then rewrite the query in the second cell so it returns the correct 3 rows.

<details><summary>Hint</summary>

The index first finds the 3 nearest notes of all patients, and only then does `WHERE` remove other patients' notes. Rank by `array_cosine_similarity(...) DESC` instead: the index does not serve it, so DuckDB filters first and then ranks the remaining rows.
</details>

In [ ]:
submit("3.4", """
CREATE INDEX IF NOT EXISTS note_hnsw ON note_vectors USING HNSW (vec) WITH (metric = 'cosine');

SELECT note_id, patient_id, note_date
FROM note_vectors
WHERE patient_id = 'P002' AND note_date >= DATE '2026-01-01'
ORDER BY array_cosine_distance(vec, getvariable('q'))
LIMIT 3;
""")

In [ ]:
submit("3.4", """
SELECT note_id, patient_id, note_date
FROM note_vectors
WHERE patient_id = 'P002' AND note_date >= DATE '2026-01-01'
ORDER BY array_cosine_distance(vec, getvariable('q'))
LIMIT 3;
""")

In [ ]:
explain("3.4", """

""")   # Why did the index return fewer than 3 rows, and why does your rewrite return the correct 3?

## 4 · Controlling the embeddings

What you embed decides what “similar” means. Each question in this section builds a new vector table, and its
check runs all five search questions against it and against `note_vectors`, side by side. Questions 4.1 and 4.2
add a search box: type any search, and it shows the top 5 notes from both tables next to each other. You can
also score a table yourself with `evaluate("table_name")`.

**Question 4.1 · Remove template text.** Most progress notes begin with the same template text (see `note_template`). Embed the notes without it into `note_vectors_clean`; the check compares the five questions' scores with `note_vectors`. Then type your own searches into the search box below and compare the two tables' results side by side.

<details><summary>Hint</summary>

Embed `replace(note, (SELECT text FROM note_template), '')` instead of `note`. In the search box, try a search for something only a few notes mention, such as “nausea”.
</details>

In [ ]:
submit("4.1", """
CREATE OR REPLACE TABLE note_vectors_clean AS
SELECT note_id, patient_id, note_date,
       embed(note) AS vec
FROM notes;
""")

In [ ]:
search_box("4.1", "note_vectors", "note_vectors_clean")   # type a search, press Search: top 5 from each table, side by side

In [ ]:
explain("4.1", """

""")   # How did removing the template change the results, and why does a repeated template matter?

**Question 4.2 · Add the note type.** Embed each note with its note type in front, for example `patient message: I keep waking up…`, into `note_vectors_typed`. Then use the search box to compare `note_vectors` and `note_vectors_typed`, including a search that names a kind of note, such as “patient messages about sleep”.

<details><summary>Hint</summary>

Embed `note_type || ': ' || note` instead of `note`.
</details>

In [ ]:
submit("4.2", """
CREATE OR REPLACE TABLE note_vectors_typed AS
SELECT note_id, patient_id, note_date,
       embed(note) AS vec
FROM notes;
""")

In [ ]:
search_box("4.2", "note_vectors", "note_vectors_typed")   # type a search, press Search: top 5 from each table, side by side

In [ ]:
explain("4.2", """

""")   # When did adding the note type help, and when did it change nothing or hurt?

**Question 4.3 · Split into sentences.** Split each note into sentences and embed each sentence into `chunk_vectors`. A note then scores as its best sentence. Compare the scores, especially for Q2: N36 is a discharge summary that mentions the low glucose only in its last sentences.

<details><summary>Hint</summary>

The `chunks` CTE already splits the notes. Replace `...` with `embed(chunk)`, and keep `note_id` so each sentence points back to its note.
</details>

In [ ]:
submit("4.3", """
CREATE OR REPLACE TABLE chunk_vectors AS
WITH chunks AS (
  SELECT note_id,
         unnest(string_split(replace(note, (SELECT text FROM note_template), ''), '. ')) AS chunk
  FROM notes
)
SELECT note_id, chunk,
       ... AS vec
FROM chunks;
""")

In [ ]:
explain("4.3", """

""")   # What did splitting into sentences gain, and what could it lose?

**Question 4.4 · What similarity misses.** Run the query: each row compares two short texts. Which pairs score high although they mean different things? Add a pair of your own.

<details><summary>Hint</summary>

Look at negation (“denies”), numbers, and abbreviations.
</details>

In [ ]:
submit("4.4", """
WITH pairs(a, b) AS (VALUES
  ('chest pain', 'denies chest pain'),
  ('chest pain', 'no complaints today'),
  ('HbA1c 8.2%', 'HbA1c 6.1%'),
  ('HbA1c 8.2%', 'HbA1c 8.3%'),
  ('shortness of breath', 'SOB')
)
SELECT a, b, array_cosine_similarity(embed(a), embed(b)) AS cosine
FROM pairs;
""")

In [ ]:
explain("4.4", """

""")   # Which pairs scored high although they mean different things, and what does that mean for searching clinical notes?

## 5 · Changing the query, and answering with the results

A three-word question is a poor match for a paragraph of clinical text. The setup cell defined three shortcuts
that call the chat model, so each question needs only a small change to a query you already have:

| Shortcut | What the chat model returns |
|---|---|
| `expand_query(text)` | other words and phrases for the same problem, separated by commas |
| `hypothetical_note(text)` | two sentences of a made-up clinical note about the problem |
| `answer_with(question, notes)` | an answer that uses only the given notes and cites their IDs |

Questions 5.1 and 5.2 compare their search with your vector search from question 3.1. Question 5.3 uses the
search to answer a question: retrieval-augmented generation (RAG).

**Question 5.1 · Query expansion.** `expand_query(text)` asks the chat model for other words and phrases for the same problem. Run the first cell to read the expansion for “trouble sleeping”, and change the text to try other questions. The second cell searches with the question stored in the variable `question`. Change `embed(getvariable('question'))` to `embed(expand_query(getvariable('question')))`, so it searches with the expansion instead. Did recall@10 improve over question 3.1?

<details><summary>Hint</summary>

Only the expression inside `embed(...)` changes. The `WITH q AS (...)` line computes the question's vector once, and the main query compares it with every note.
</details>

In [ ]:
submit("5.1", """
SELECT expand_query('trouble sleeping') AS expansion;
""")

In [ ]:
submit("5.1", """
SET VARIABLE question = 'trouble sleeping';

WITH q AS (SELECT embed(getvariable('question')) AS qv)
SELECT n.note_id, n.note, array_cosine_similarity(v.vec, q.qv) AS similarity
FROM note_vectors AS v JOIN notes AS n USING (note_id), q
ORDER BY similarity DESC
LIMIT 10;
""")

In [ ]:
explain("5.1", """

""")   # Did the expansion help or hurt, and which phrases in it made the difference?

**Question 5.2 · Hypothetical note.** `hypothetical_note(text)` asks the chat model to write a short made-up note about the problem. Searching with that note's vector instead of the question's is called hypothetical document embeddings (HyDE). Run the first cell to read a hypothetical note, and try other questions. Then change the second cell to `embed(hypothetical_note(getvariable('question')))`. Why can a made-up note find real notes better than the question?

<details><summary>Hint</summary>

A note is closer in style and vocabulary to other notes than a three-word question is. The made-up note's facts do not matter; only its vector is used.
</details>

In [ ]:
submit("5.2", """
SELECT hypothetical_note('trouble sleeping') AS hypothetical_note;
""")

In [ ]:
submit("5.2", """
SET VARIABLE question = 'trouble sleeping';

WITH q AS (SELECT embed(getvariable('question')) AS qv)
SELECT n.note_id, n.note, array_cosine_similarity(v.vec, q.qv) AS similarity
FROM note_vectors AS v JOIN notes AS n USING (note_id), q
ORDER BY similarity DESC
LIMIT 10;
""")

In [ ]:
explain("5.2", """

""")   # How did searching with a made-up note compare with the question and with the expansion?

**Question 5.3 · Answer with the notes (RAG).** Retrieval-augmented generation (RAG) answers a question from retrieved text. The query finds P002's 5 notes most similar to the question. Finish it: join each note's ID and text into one string with `string_agg`, and pass the question and that string to `answer_with(question, notes)`, which asks the chat model to answer using only those notes and to cite their IDs. Is the answer right, and does it cite the right notes?

<details><summary>Hint</summary>

`SELECT answer_with('Has this patient had trouble sleeping?', string_agg(note_id || ': ' || note, '\n')) AS answer FROM top_notes;`. The patient filter runs before the ranking, so only P002's notes can reach the model.
</details>

In [ ]:
submit("5.3", """
SET VARIABLE q = (SELECT embed('Has this patient had trouble sleeping?'));

WITH top_notes AS (
  SELECT n.note_id, n.note
  FROM note_vectors AS v JOIN notes AS n USING (note_id)
  WHERE v.patient_id = 'P002'
  ORDER BY array_cosine_similarity(v.vec, getvariable('q')) DESC
  LIMIT 5
)
SELECT ...;
""")

In [ ]:
explain("5.3", """

""")   # Is the answer supported by the notes it cites? Did any retrieved note mislead it?

## 6 · Submit your notebook

Submit this notebook with its outputs: in Colab, **File > Download > Download .ipynb**, then upload that file.
The outputs under each cell (your results and the checks) are part of what is graded, so run every `submit`
and `explain` cell you edited before you download. `progress()` lists which questions you have submitted and
answered in words.

In [ ]:
progress()